# Multi-Scale All-CNN Hybrid Extractor for Diabetic Retinopathy Grading

This notebook contains the complete PyTorch implementation of the **Multi-Scale All-CNN Hybrid Extractor** architecture designed for 5-class Diabetic Retinopathy (DR) grading using a Unilateral (single-eye) approach.

### Key Architecture Components:
1. **Global Context Branch**: Pre-trained `ResNet50` + `CBAM` Attention $\rightarrow F_{\text{global}} \in \mathbb{R}^{2048}$
2. **Local Patch Branch**: Pre-trained `EfficientNet-B0` + Reshape Trick + Patch Max-Pooling $\rightarrow F_{\text{local}} \in \mathbb{R}^{1280}$
3. **Intra-Fusion & MLP**: Feature concatenation ($3328$-dim) $\rightarrow$ 2-layer MLP $\rightarrow 512$-dim embedding
4. **Ordinal Classification Head**: Consistent Rank Logits (CORAL) outputting 4 binary cumulative logits $P(y > k)$
5. **Custom Loss Module**: `HybridOrdinalQWKLoss` combining Weighted Cross-Entropy ($L_{\text{WCE}}$), Ordinal BCE ($L_{\text{Ordinal}}$), and Soft QWK Loss ($L_{\text{QWK}}$)
6. **Smart Crop Utility**: High-contrast sliding window intensity variance patch extraction (`smart_crop_patches`)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
from PIL import Image
from typing import List, Tuple, Union

print(f"PyTorch Version: {torch.__version__}")

## 1. Convolutional Block Attention Module (CBAM)

CBAM sequentially applies **Channel Attention** (combining Average Pooling and Max Pooling through a shared MLP) and **Spatial Attention** ($7\times7$ convolution over spatial channel projections).

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_planes: int, reduction_ratio: int = 16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        hidden_planes = max(in_planes // reduction_ratio, 8)
        self.mlp = nn.Sequential(
            nn.Linear(in_planes, hidden_planes, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_planes, in_planes, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.size()
        avg_out = self.mlp(self.avg_pool(x).view(b, c))
        max_out = self.mlp(self.max_pool(x).view(b, c))
        out = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return out

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super(SpatialAttention, self).__init__()
        assert kernel_size in (3, 7), "Kernel size must be 3 or 7"
        padding = 3 if kernel_size == 7 else 1
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)
        out = self.sigmoid(self.conv(concat))
        return out

class CBAM(nn.Module):
    def __init__(self, gate_channels: int, reduction_ratio: int = 16, spatial_kernel_size: int = 7):
        super(CBAM, self).__init__()
        self.channel_att = ChannelAttention(gate_channels, reduction_ratio=reduction_ratio)
        self.spatial_att = SpatialAttention(kernel_size=spatial_kernel_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x

## 2. Global Context Branch (Zoom-out)

Processes whole-fundus images `[Batch, 3, 224, 224]` through pre-trained `ResNet50` feature extractor + `CBAM` attention $\rightarrow F_{\text{global}} \in \mathbb{R}^{\text{Batch} \times 2048}$.

In [ ]:
class GlobalBranch(nn.Module):
    def __init__(self, pretrained: bool = True):
        super(GlobalBranch, self).__init__()
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        resnet = models.resnet50(weights=weights)
        
        # Extract feature maps up to layer4
        self.backbone = nn.Sequential(*list(resnet.children())[:-2]) # [B, 2048, 7, 7]
        self.cbam = CBAM(gate_channels=2048, reduction_ratio=16)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, img_global: torch.Tensor) -> torch.Tensor:
        # img_global: [Batch, 3, 224, 224]
        feat = self.backbone(img_global)    # [Batch, 2048, 7, 7]
        feat = self.cbam(feat)              # [Batch, 2048, 7, 7]
        feat = self.global_pool(feat)       # [Batch, 2048, 1, 1]
        F_global = torch.flatten(feat, 1)   # [Batch, 2048]
        return F_global

## 3. Local Patch Branch (Zoom-in)

Processes $K$ high-resolution patches `[Batch, K, 3, 128, 128]` using `EfficientNet-B0`. Applies reshape trick (`Batch*K`), unflattens back to `[Batch, K, 1280]`, and performs **MaxPool** across patch dimension $K \rightarrow F_{\text{local}} \in \mathbb{R}^{\text{Batch} \times 1280}$.

In [ ]:
class LocalBranch(nn.Module):
    def __init__(self, pretrained: bool = True):
        super(LocalBranch, self).__init__()
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        effnet = models.efficientnet_b0(weights=weights)
        
        self.features = effnet.features      # [B*K, 1280, 4, 4]
        self.avgpool = effnet.avgpool        # [B*K, 1280, 1, 1]

    def forward(self, patches_local: torch.Tensor) -> torch.Tensor:
        # patches_local: [Batch, K, 3, 128, 128]
        batch_size, K, C, H, W = patches_local.shape
        
        # Reshape trick: merge Batch and K dimensions
        x = patches_local.view(batch_size * K, C, H, W)  # [Batch * K, 3, 128, 128]
        
        feat = self.features(x)              # [Batch * K, 1280, 4, 4]
        feat = self.avgpool(feat)            # [Batch * K, 1280, 1, 1]
        feat = torch.flatten(feat, 1)        # [Batch * K, 1280]
        
        # Unflatten back to [Batch, K, 1280]
        feat = feat.view(batch_size, K, -1)  # [Batch, K, 1280]
        
        # Max aggregation across K patches
        F_local, _ = torch.max(feat, dim=1)  # [Batch, 1280]
        return F_local

## 4. Ordinal Classification Head (CORAL Framework)

Outputs 4 binary logits representing cumulative threshold probabilities $P(y > 0), P(y > 1), P(y > 2), P(y > 3)$ and reconstructs discrete 5-class nominal probabilities $P(y = k)$.

In [ ]:
class OrdinalHead(nn.Module):
    def __init__(self, in_features: int = 512, num_classes: int = 5):
        super(OrdinalHead, self).__init__()
        self.num_classes = num_classes
        self.num_logits = num_classes - 1
        self.fc = nn.Linear(in_features, self.num_logits)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [Batch, 512]
        logits = self.fc(x)  # [Batch, 4]
        return logits

    @staticmethod
    def logits_to_class_probs(logits: torch.Tensor) -> torch.Tensor:
        cum_probs = torch.sigmoid(logits)  # [Batch, 4]
        p_gt0 = cum_probs[:, 0:1]
        p_gt1 = cum_probs[:, 1:2]
        p_gt2 = cum_probs[:, 2:3]
        p_gt3 = cum_probs[:, 3:4]
        
        p0 = 1.0 - p_gt0
        p1 = p_gt0 - p_gt1
        p2 = p_gt1 - p_gt2
        p3 = p_gt2 - p_gt3
        p4 = p_gt3
        
        probs = torch.cat([p0, p1, p2, p3, p4], dim=1)  # [Batch, 5]
        probs = torch.clamp(probs, min=1e-7, max=1.0)
        probs = probs / probs.sum(dim=1, keepdim=True)
        return probs

## 5. Main Model Architecture (`MultiScaleHybridExtractor`)

Integrates `GlobalBranch` ($2048$-dim) and `LocalBranch` ($1280$-dim), concatenates into $3328$-dim, passes through a 2-layer MLP ($3328 \to 1024 \to 512$), and feeds into `OrdinalHead`.

In [ ]:
class MultiScaleHybridExtractor(nn.Module):
    def __init__(self, pretrained: bool = True, num_classes: int = 5):
        super(MultiScaleHybridExtractor, self).__init__()
        self.global_branch = GlobalBranch(pretrained=pretrained)
        self.local_branch = LocalBranch(pretrained=pretrained)
        
        fused_dim = 2048 + 1280  # 3328-dim
        
        self.mlp = nn.Sequential(
            nn.Linear(fused_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True)
        )
        
        self.ordinal_head = OrdinalHead(in_features=512, num_classes=num_classes)

    def forward(self, img_global: torch.Tensor, patches_local: torch.Tensor) -> torch.Tensor:
        # Step A: Global Branch -> [Batch, 2048]
        F_global = self.global_branch(img_global)
        
        # Step B: Local Patch Branch -> [Batch, 1280]
        F_local = self.local_branch(patches_local)
        
        # Step C: Fusion MLP -> [Batch, 512]
        F_fused = torch.cat([F_global, F_local], dim=1)     # [Batch, 3328]
        F_latent = self.mlp(F_fused)                        # [Batch, 512]
        
        # Step D: CORAL Ordinal Head -> [Batch, 4]
        logits = self.ordinal_head(F_latent)
        return logits

## 6. Custom Hybrid Loss Function (`HybridOrdinalQWKLoss`)

Calculates composite loss: $\text{Loss} = \alpha L_{\text{WCE}} + \beta L_{\text{Ordinal}} + \gamma L_{\text{QWK}}$.

In [ ]:
class HybridOrdinalQWKLoss(nn.Module):
    def __init__(self, alpha: float = 1.0, beta: float = 1.0, gamma: float = 1.0,
                 class_weights: Union[torch.Tensor, None] = None, num_classes: int = 5):
        super(HybridOrdinalQWKLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.num_classes = num_classes
        
        self.bce_loss = nn.BCEWithLogitsLoss()
        if class_weights is not None:
            self.register_buffer('class_weights', class_weights)
        else:
            self.class_weights = None

    def _ordinal_targets(self, targets: torch.Tensor) -> torch.Tensor:
        batch_size = targets.size(0)
        num_logits = self.num_classes - 1
        ranks = torch.arange(num_logits, device=targets.device).unsqueeze(0).expand(batch_size, num_logits)
        targets_expanded = targets.unsqueeze(1).expand(batch_size, num_logits)
        ordinal_targets = (targets_expanded > ranks).float()
        return ordinal_targets

    def _soft_qwk_loss(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        cum_probs = torch.sigmoid(logits)
        pred_rank = torch.sum(cum_probs, dim=1)
        target_rank = targets.float()
        max_dist_sq = (self.num_classes - 1) ** 2
        mse = torch.mean((pred_rank - target_rank) ** 2)
        return mse / max_dist_sq

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        # 1. L_Ordinal
        ord_targets = self._ordinal_targets(targets)
        l_ordinal = self.bce_loss(logits, ord_targets)
        
        # 2. L_WCE
        class_probs = OrdinalHead.logits_to_class_probs(logits)
        log_probs = torch.log(class_probs)
        if self.class_weights is not None:
            l_wce = F.nll_loss(log_probs, targets, weight=self.class_weights)
        else:
            l_wce = F.nll_loss(log_probs, targets)
            
        # 3. L_QWK
        l_qwk = self._soft_qwk_loss(logits, targets)
        
        total_loss = (self.alpha * l_wce) + (self.beta * l_ordinal) + (self.gamma * l_qwk)
        return total_loss, l_wce, l_ordinal, l_qwk

## 7. Smart Crop Utility (`smart_crop_patches`)

Scans high-resolution fundus images using sliding windows, ranks patches by intensity variance, and applies Non-Maximum Suppression to extract $K$ high-contrast lesion patches.

In [ ]:
def smart_crop_patches(
    image: Union[Image.Image, np.ndarray],
    K: int = 4,
    patch_size: int = 128
) -> List[np.ndarray]:
    if isinstance(image, Image.Image):
        img_np = np.array(image)
    else:
        img_np = image.copy()
        
    H, W, C = img_np.shape
    assert H >= patch_size and W >= patch_size

    if C == 3:
        gray = np.dot(img_np[..., :3], [0.2989, 0.5870, 0.1140])
    else:
        gray = img_np.squeeze()

    step = patch_size // 2
    candidate_windows = []

    for y in range(0, H - patch_size + 1, step):
        for x in range(0, W - patch_size + 1, step):
            win_gray = gray[y : y + patch_size, x : x + patch_size]
            var_score = np.var(win_gray)
            candidate_windows.append((var_score, y, x))

    candidate_windows.sort(key=lambda item: item[0], reverse=True)

    selected_patches = []
    selected_boxes = []

    for score, y, x in candidate_windows:
        if len(selected_patches) >= K:
            break
        overlap = False
        for (sy, sx) in selected_boxes:
            if abs(y - sy) < step and abs(x - sx) < step:
                overlap = True
                break
        if not overlap:
            patch = img_np[y : y + patch_size, x : x + patch_size, :]
            selected_patches.append(patch)
            selected_boxes.append((y, x))

    while len(selected_patches) < K:
        y = np.random.randint(0, H - patch_size + 1)
        x = np.random.randint(0, W - patch_size + 1)
        patch = img_np[y : y + patch_size, x : x + patch_size, :]
        selected_patches.append(patch)

    return selected_patches

## 8. Verification & Execution Test

Runs dummy tensors through `MultiScaleHybridExtractor`, `HybridOrdinalQWKLoss`, backpropagation, and `smart_crop_patches` to verify complete execution.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing verification test on device: {device}")

torch.manual_seed(42)
np.random.seed(42)

Batch_Size = 2
K_patches = 4

# 1. Instantiate Model
model = MultiScaleHybridExtractor(pretrained=False, num_classes=5).to(device)
model.eval()

dummy_img_global = torch.randn(Batch_Size, 3, 224, 224, device=device)
dummy_patches_local = torch.randn(Batch_Size, K_patches, 3, 128, 128, device=device)
dummy_targets = torch.tensor([1, 4], device=device, dtype=torch.long)

# 2. Forward Pass
with torch.no_grad():
    logits = model(dummy_img_global, dummy_patches_local)
    class_probs = OrdinalHead.logits_to_class_probs(logits)

print(f"Logits Shape:            {logits.shape}")
print(f"Class Probs Shape:       {class_probs.shape}")
print(f"Class Probs Sums:        {class_probs.sum(dim=1).cpu().numpy()} (Should be 1.0)")

# 3. Hybrid Loss & Backprop
weights = torch.tensor([1.0, 2.0, 1.5, 3.0, 4.0], device=device)
criterion = HybridOrdinalQWKLoss(alpha=1.0, beta=1.0, gamma=1.0, class_weights=weights).to(device)

model.train()
logits_train = model(dummy_img_global, dummy_patches_local)
loss, l_wce, l_ordinal, l_qwk = criterion(logits_train, dummy_targets)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
optimizer.zero_grad()
loss.backward()
optimizer.step()

print(f"Total Loss:              {loss.item():.4f}")
print(f"Backward pass successful! Parameter gradients computed.")

# 4. Smart Crop Utility
dummy_pil_img = Image.fromarray(np.random.randint(0, 256, (1024, 1024, 3), dtype=np.uint8))
extracted_patches = smart_crop_patches(dummy_pil_img, K=K_patches, patch_size=128)
print(f"Extracted Patches Count: {len(extracted_patches)}")
print(f"Individual Patch Shape:  {extracted_patches[0].shape}")

print("\nALL TESTS PASSED SUCCESSFULLY!")